# L08 · Actor-Critic·GAE·PPO from scratch

## Goal

- critic과 GAE를 계산한다
- old/current ratio를 구분한다
- PPO clip의 advantage 부호를 검산한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L08:toy:42").hexdigest()
print(f"lesson=L08 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L08 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:f21ea10a7ab463f89b492a9a914c94806bcb3fb04b187ae947a27554b31f76be data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: REINFORCE → **Actor-Critic·GAE·PPO** → LLM policy

$$\hat A_t=\delta_t+(\gamma\lambda)\hat A_{t+1},\qquad L^{clip}=\min(r_t\hat A_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)\hat A_t)$$

critic은 state value를 예측하고 TD residual `δ`가 advantage의 재료가 됩니다. GAE의 lambda는 짧은 bootstrap과 긴 return 사이를 잇습니다. PPO ratio는 rollout 당시 old policy와 현재 policy의 선택확률 비율입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** ratio=1.25, epsilon=0.2일 때 positive와 negative advantage 모두 같은 방식으로 잘릴까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아닙니다. `min` 때문에 positive advantage는 1.2에서 제한되지만 negative advantage는 더 나쁜 -1.25가 선택됩니다. 양쪽 부호를 따로 검산해야 합니다.</details>

In [2]:
from rl_study.algorithms.ppo import ppo_policy_loss
from rl_study.math import generalized_advantage_estimate
rewards = torch.tensor([0.0, 1.0])
values = torch.tensor([0.2, 0.4, 0.0])
terminated = torch.tensor([False, True])
truncated = torch.tensor([False, False])
gae, gae_returns = generalized_advantage_estimate(
    rewards, values, terminated, truncated, gamma=0.9, gae_lambda=0.95
)
old_logp = torch.zeros(2)
current_logp = torch.log(torch.tensor([1.25, 1.25]))
ppo_output = ppo_policy_loss(current_logp, old_logp, torch.tensor([1.0, -1.0]))
print({"gae": gae.tolist(), "returns": gae_returns.tolist(),
       "ratios": ppo_output.ratio.tolist(),
       "clipped": ppo_output.clipped_objective.tolist()})

{'gae': [0.6729999780654907, 0.6000000238418579], 'returns': [0.8729999661445618, 1.0], 'ratios': [1.25, 1.25], 'clipped': [1.2000000476837158, -1.2000000476837158]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** old log-prob는 rollout snapshot이고 detach되어야 합니다. KL penalty나 early stopping도 대안이며, clip 하나가 실제 trust region을 보장한다고 해석해서는 안 됩니다.

**흔한 함정:** current policy로 old log-prob를 다시 계산하면 ratio가 늘 1이 되어 update 진단이 무력화됩니다. rollout artifact의 policy version과 log-prob를 보존합니다. 회귀 test: `test_gae_analytic`, `test_ppo_clip_sign_cases`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert torch.allclose(ppo_output.ratio, torch.tensor([1.25, 1.25]))
assert torch.isfinite(gae).all()
print("checks=passed")

checks=passed


**회상 문제:** lambda=0과 lambda=1은 각각 어떤 target에 가까워지나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** GAE는 유한한 두 advantage를 만들었고 ratio는 둘 다 1.25였습니다. 출력의 clipped 항은 advantage 부호에 따라 비대칭입니다.
- 실제 확인: `test_gae_analytic`, `test_ppo_clip_sign_cases`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L09에서 PPO의 한 action을 LLM response token들로 바꾸고 KL·mask·reward 위치를 다시 정의합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `gae-2015` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `repo-spinningup` — `docs/sources.yml`